<h3>Runnable & Runnable Passthrough</h3>

Runnable is anything you can “pipe” (|) data through (prompts, LLMs, retrievers, functions, etc.).
RunnablePassthrough is a tiny helper Runnable that forwards its input unchanged. It’s most useful when you want to:

Preserve the original input while also computing extra fields derived from that input.
Branch: feed the same input to multiple sub-steps (e.g., to an LLM and to a function) and then recombine.
Enrich: add keys to an object (dict) on-the-fly without losing existing keys.

Think of it as “keep the input as-is, but let me attach more stuff next to it.”
There are two very common patterns:

RunnablePassthrough() — just forwards input.
RunnablePassthrough.assign(key1=..., key2=...) — forwards input and adds/overwrites fields using other runnables/functions.

There are two very common patterns:

RunnablePassthrough() — just forwards input.
RunnablePassthrough.assign(key1=..., key2=...) — forwards input and adds/overwrites fields using other runnables/functions.

In [ ]:
# Simple passthrough runnable example
from langchain_core.runnables import RunnablePassthrough

pt = RunnablePassthrough()

print(pt.invoke("Hello"))       # -> "Hello"
print(pt.invoke({"x": 1}))      # -> {"x": 1}


Hello
{'x': 1}


In [ ]:
# Example of a more complex runnable that adds 1 to a number
from langchain_core.runnables import Runnable

class AddOneRunnable(Runnable):
    def invoke(self, input):
        return input + 1

add_one = AddOneRunnable()
print(add_one.invoke(5))  # -> 6

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from azure_openai_llm import create_azure_llm

llm = create_azure_llm()

def extract_city(user_text: str) -> str:
    for city in ["Paris", "Riyadh", "Jeddah", "Lahore"]:
        if city.lower() in user_text.lower():
            return city
    return "Unknown"

# 1) Build a prompt that expects both 'question' and 'city'
prompt = ChatPromptTemplate.from_template(
    "Answer the user's question.\n"
    "City (detected): {city}\n"
    "Question: {question}"
)

# 2) Build a small preprocessor that turns the input string into a dict {question: <str>} and enriches it
pre = (
    # Turn the raw string into a mapping with a 'question' key
    (lambda x: {"question": x})
    # Add 'city' while preserving 'question'
    | RunnablePassthrough.assign(city=lambda d: extract_city(d["question"]))
)

# 3) Compose the full chain
chain = pre | prompt | llm

print(chain.invoke("What's the weather in Paris today?").content)

Using chat deployment: lums-gpt-4.1-mini
Endpoint: https://aoai-foundry-swc.openai.azure.com/
API Version: 2025-01-01-preview
I don't have real-time access to current weather data. For the latest weather updates in Paris today, please check a reliable weather website or app like Weather.com, AccuWeather, or a local news source.
